# NDP Endpoint Tutorial: General Dataset Management

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/national-data-platform/ep-api/blob/main/docs/general_dataset_api_tutorial.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/national-data-platform/ep-api/main?filepath=docs/general_dataset_api_tutorial.ipynb)

This notebook shows how to manage datasets in the local catalog of an
NDP Endpoint through its API. You will:

1. Check the Endpoint and authenticate with a bearer token
2. Create an organization to hold the datasets
3. Create a dataset with tags, custom metadata (`extras`) and resources
4. Read it back, update it partially (`PATCH`) and fully (`PUT`)
5. Update and remove individual resources
6. See how the API reports validation errors
7. Delete everything the notebook created

Every object the notebook creates carries a unique run id and is
deleted in the last section, so it is safe to run against a shared
Endpoint.

## Prerequisites

- Python 3.11+ and the `requests` library
- An NDP Endpoint with a local catalog (`LOCAL_CATALOG_BACKEND` set to `ckan` or `mongodb`)
- A token that may write on that Endpoint (see the configuration section)

In [ ]:
# Install the only dependency (already present on most Jupyter setups)
%pip install -q requests

## 1. Configuration

The notebook reads two environment variables so it can run unchanged
against any Endpoint:

| Variable | Default | Meaning |
|----------|---------|---------|
| `EP_API_URL` | `http://localhost:8002` | Base URL of the Endpoint API (8002 is the port published by the bundled `docker-compose.yml`). |
| `EP_TOKEN` | `testing_token` | Bearer token sent in the `Authorization` header. |

> **About the token:** `testing_token` is the development `TEST_TOKEN`
> that a local Endpoint accepts when it is started with `TEST_TOKEN=testing_token`.
> A real Endpoint does not accept it: export `EP_TOKEN` with your NDP
> access token instead (write operations need a token whose user may
> write on this Endpoint).

In [ ]:
import datetime
import json
import os
import uuid

import requests

API_BASE_URL = os.getenv("EP_API_URL", "http://localhost:8002").rstrip("/")
TOKEN = os.getenv("EP_TOKEN", "testing_token")
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

# A unique suffix for everything this notebook creates, so it never
# collides with existing data and can be cleaned up precisely.
RUN_ID = (
    datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
    + uuid.uuid4().hex[:6]
)

print(f"API base URL: {API_BASE_URL}")
print(f"Run id:       {RUN_ID}")

### Helper functions

`call()` sends the token on every request, prints the method, path and
status code, and stops the notebook on an unexpected status.

In [ ]:
def call(method, path, expected=(200, 201), **kwargs):
    """
    Send an authenticated request to the Endpoint and return (status, body).

    Raises RuntimeError when the status code is not in `expected`, so a
    failing step stops the notebook instead of silently cascading.
    Pass `expected=None` to accept any status (used for error demos).
    """
    headers = {**HEADERS, **kwargs.pop("headers", {})}
    kwargs.setdefault("timeout", 60)
    response = requests.request(
        method, f"{API_BASE_URL}{path}", headers=headers, **kwargs
    )
    try:
        body = response.json()
    except ValueError:
        body = response.text
    print(f"{method} {path} -> {response.status_code}")
    if expected is not None and response.status_code not in expected:
        raise RuntimeError(
            f"Unexpected status {response.status_code}: {json.dumps(body, indent=2)}"
        )
    return response.status_code, body


def show(data):
    """Pretty-print a JSON-like value."""
    print(json.dumps(data, indent=2, default=str))

## 2. Check the Endpoint

`GET /status/` (note the trailing slash) needs a token and reports
the API version and which optional features this Endpoint has
enabled.

In [ ]:
_, status = call("GET", "/status/")
for key in ("api_version", "local_catalog_backend", "backend_connected", "group_based_access", "s3_enabled", "kafka_enabled"):
    print(f"{key:24} {status.get(key)}")

In [ ]:
if not status.get("backend_connected"):
    raise RuntimeError(
        "The local catalog is not reachable; this tutorial needs it."
    )

## 3. Organizations

Every dataset belongs to an organization. `GET /organization` lists the
organization names of the selected catalog (`server=local` is the
Endpoint's own catalog).

In [ ]:
_, organizations = call("GET", "/organization", params={"server": "local"})
print(f"{len(organizations)} organizations: {organizations[:10]}")

Create an organization for this tutorial. Names must be lowercase
letters, digits, `-` or `_`.

In [ ]:
ORG_NAME = f"tutorial_datasets_{RUN_ID}"

_, org = call(
    "POST",
    "/organization",
    params={"server": "local"},
    json={
        "name": ORG_NAME,
        "title": f"Tutorial datasets {RUN_ID}",
        "description": "Temporary organization created by the general dataset tutorial.",
    },
)
show(org)

## 4. Create a dataset

`POST /dataset` takes `name`, `title` and `owner_org` (required) plus
optional `notes`, `tags`, `extras` (free key/value metadata),
`resources`, `private`, `license_id` and `version`.

Notes:

- Each resource needs a `url` and a `name`; `format`, `description`,
  `mimetype` and `size` are optional.
- Tags must be at least two characters long.
- `groups` is also accepted, but it only links the dataset to catalog
  groups that already exist, so it is left out here.
- Some keys are reserved and cannot be used in `extras` (for example
  `name`, `id`, `title`, `resources`).

In [ ]:
DATASET_NAME = f"tutorial_climate_{RUN_ID}"

dataset_payload = {
    "name": DATASET_NAME,
    "title": "Climate Research Dataset (tutorial)",
    "owner_org": ORG_NAME,
    "notes": "Temperature and precipitation measurements, created by the general dataset tutorial.",
    "tags": ["climate", "temperature", "tutorial"],
    "license_id": "cc-by",
    "version": "1.0",
    "private": False,
    "extras": {
        "project_code": "CLIMATE-TUTORIAL",
        "temporal_resolution": "daily",
        "geographical_coverage": "North America",
    },
    "resources": [
        {
            "url": "https://data.example.org/climate/daily_temperature.csv",
            "name": "daily_temperature",
            "format": "CSV",
            "description": "Daily minimum, maximum and mean temperature.",
            "mimetype": "text/csv",
            "size": 15728640,
        },
        {
            "url": "https://data.example.org/climate/monthly_precipitation.json",
            "name": "monthly_precipitation",
            "format": "JSON",
            "description": "Monthly precipitation totals.",
            "mimetype": "application/json",
        },
    ],
}

_, created = call("POST", "/dataset", params={"server": "local"}, json=dataset_payload)
show(created)

DATASET_ID = created["id"]
# Everything recorded here is deleted at the end of the notebook
created_dataset_ids = [DATASET_ID]
created_dataset_names = [created["name"]]

The response contains the dataset `id`, the final `name` and `title`,
and a `warning`. If the name is already taken, the API does **not**
fail: it saves the dataset under a timestamped name and explains the
rename in `warning`. Always use the returned `name`/`id`, not the one
you sent:

In [ ]:
_, duplicate = call(
    "POST",
    "/dataset",
    params={"server": "local"},
    json={"name": DATASET_NAME, "title": "Same name again", "owner_org": ORG_NAME},
)
show(duplicate)
created_dataset_ids.append(duplicate["id"])
created_dataset_names.append(duplicate["name"])

## 5. Read the dataset back

`GET /search` takes one or more `terms` and returns matching datasets
with their resources and extras.

In [ ]:
def find_dataset(name):
    _, results = call("GET", "/search", params={"terms": name, "server": "local"})
    return next((d for d in results if d["name"] == name), None)


dataset = find_dataset(DATASET_NAME)
print("title:   ", dataset["title"])
print("org:     ", dataset["owner_org"])
print("extras:  ", {k: v for k, v in dataset["extras"].items() if not k.startswith("ndp_")})
for resource in dataset["resources"]:
    print(f"resource: {resource['name']} ({resource['format']}) -> {resource['url']}")

The Endpoint adds `ndp_*` extras (who created the dataset and for which
organization) automatically; you never send them yourself.

A single resource can be fetched by id with `GET /resource/{resource_id}`:

In [ ]:
RESOURCE_ID = dataset["resources"][0]["id"]
_, resource = call("GET", f"/resource/{RESOURCE_ID}", params={"server": "local"})
print({k: resource.get(k) for k in ("name", "format", "mimetype", "size", "url")})

## 6. Update the dataset

### Partial update (`PATCH /dataset/{id}`)

Only the fields you send change. `extras` are merged with the existing
ones, and `resources` are **added** (a resource with the same `url` or
`name` as an existing one updates it instead).

In [ ]:
call(
    "PATCH",
    f"/dataset/{DATASET_ID}",
    params={"server": "local"},
    json={
        "version": "1.1",
        "extras": {"quality_level": "validated"},
        "resources": [
            {
                "url": "https://data.example.org/climate/stations.geojson",
                "name": "stations",
                "format": "GeoJSON",
                "description": "Location of the weather stations.",
            }
        ],
    },
)

dataset = find_dataset(DATASET_NAME)
print("extras:   ", sorted(k for k in dataset["extras"] if not k.startswith("ndp_")))
print("resources:", [r["name"] for r in dataset["resources"]])

### Full update (`PUT /dataset/{id}`)

`PUT` replaces the fields you send. In particular the `resources` list
you send **replaces** all existing resources, and `tags` replace the
existing tags. `extras` are still merged.

In [ ]:
call(
    "PUT",
    f"/dataset/{DATASET_ID}",
    params={"server": "local"},
    json={
        "title": "Climate Research Dataset (tutorial, v2)",
        "notes": "Second edition with a single consolidated file.",
        "tags": ["climate", "tutorial", "v2"],
        "version": "2.0",
        "license_id": "cc-by-sa",
        "resources": [
            {
                "url": "https://data.example.org/climate/climate_v2.parquet",
                "name": "climate_v2",
                "format": "Parquet",
                "description": "All variables in one file.",
            }
        ],
    },
)

dataset = find_dataset(DATASET_NAME)
print("title:    ", dataset["title"])
print("resources:", [r["name"] for r in dataset["resources"]])

## 7. Work with individual resources

- `PATCH /dataset/{dataset_id}/resource/{resource_id}` changes one
  resource (`name`, `url`, `description`, `format`).
- `DELETE /dataset/{dataset_id}/resource/{resource_id}` removes one
  resource from the dataset.

Add a second resource first, then edit and delete it.

In [ ]:
call(
    "PATCH",
    f"/dataset/{DATASET_ID}",
    params={"server": "local"},
    json={"resources": [{"url": "https://data.example.org/climate/readme.txt", "name": "readme", "format": "TXT"}]},
)
dataset = find_dataset(DATASET_NAME)
readme = next(r for r in dataset["resources"] if r["name"] == "readme")

_, patched = call(
    "PATCH",
    f"/dataset/{DATASET_ID}/resource/{readme['id']}",
    params={"server": "local"},
    json={"description": "How to cite and use this dataset."},
)
print("patched description:", patched["description"])

call("DELETE", f"/dataset/{DATASET_ID}/resource/{readme['id']}", params={"server": "local"})
print("resources now:", [r["name"] for r in find_dataset(DATASET_NAME)["resources"]])

## 8. Validation errors

Invalid requests are rejected before anything is written:

- a missing required field gives `422` with the offending fields,
- a resource without `url`/`name` gives `422`,
- reserved keys in `extras` give `400`.

In [ ]:
bad_requests = {
    "missing name and owner_org": {"title": "No name"},
    "resource without url/name": {
        "name": f"tutorial_invalid_{RUN_ID}",
        "title": "Invalid resource",
        "owner_org": ORG_NAME,
        "resources": [{"resource_url": "https://example.org/data.csv"}],
    },
    "reserved keys in extras": {
        "name": f"tutorial_reserved_{RUN_ID}",
        "title": "Reserved keys",
        "owner_org": ORG_NAME,
        "extras": {"name": "clash", "id": "clash", "custom_field": "fine"},
    },
}

for label, payload in bad_requests.items():
    status_code, body = call("POST", "/dataset", expected=None, json=payload)
    print(f"  {label}: {status_code} {body.get('detail')}\n")

## 9. Creating several datasets

A small helper that builds datasets from a template and records their
ids for cleanup.

In [ ]:
def create_from_template(suffix, title, tags):
    payload = {
        "name": f"tutorial_sample_{suffix}_{RUN_ID}",
        "title": title,
        "owner_org": ORG_NAME,
        "notes": f"Sample dataset '{title}' created by the tutorial.",
        "license_id": "cc-by",
        "tags": ["tutorial", *tags],
        "extras": {"created_via": "general dataset tutorial"},
    }
    _, body = call("POST", "/dataset", params={"server": "local"}, json=payload)
    created_dataset_ids.append(body["id"])
    created_dataset_names.append(body["name"])
    return body["name"]


for suffix, title, tags in [
    ("humidity", "Sample humidity data", ["humidity"]),
    ("wind", "Sample wind data", ["wind"]),
]:
    print("created", create_from_template(suffix, title, tags))

## 10. Clean up

Deleting needs a token that may write on the Endpoint. The API offers:

- `DELETE /resource?resource_id={id}` deletes a dataset by id,
- `DELETE /resource/{name}` deletes a dataset by name,
- `DELETE /organization/{name}?cascade=true` deletes an organization
  together with any datasets still in it (without `cascade=true` it
  answers `409` while the organization still has datasets).

Deletes are permanent (the dataset is purged from the catalog).

In [ ]:
# Delete the main dataset by name and the others by id
call("DELETE", f"/resource/{DATASET_NAME}", params={"server": "local"})
for dataset_id in created_dataset_ids[1:]:
    call("DELETE", "/resource", params={"resource_id": dataset_id, "server": "local"})

# Remove the organization (cascade also removes anything left behind)
_, deleted = call(
    "DELETE", f"/organization/{ORG_NAME}", params={"server": "local", "cascade": "true"}
)
print(deleted["message"])

In [ ]:
# Verify that nothing created by this run is left
leftovers = [name for name in created_dataset_names if find_dataset(name)]
_, organizations = call("GET", "/organization", params={"server": "local"})
print("datasets left:", leftovers)
print("organization left:", ORG_NAME in organizations)
assert not leftovers and ORG_NAME not in organizations

## Summary

You created an organization and datasets with `POST /organization` and
`POST /dataset`, read them with `GET /search` and `GET /resource/{id}`,
updated them with `PATCH`/`PUT /dataset/{id}` and the per-resource
routes, and deleted everything again.

Related tutorials:

- [S3 storage](./s3_api_tutorial.ipynb) and the
  [S3 to dataset workflow](./s3_to_dataset_workflow_tutorial.ipynb)
- [Registering and proxying services](./dogs_service_api_tutorial.ipynb)
- [Pelican federations](./pelican_api_tutorial.ipynb)

Full reference: the Endpoint's interactive API docs at `<EP_API_URL>/docs`
and the [ep-api repository](https://github.com/national-data-platform/ep-api).